In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [4]:
base = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'

print("=== genres_stems ===")
for item in os.listdir(base + '/genres_stems')[:10]:
    print(item)

print("\n=== mashups ===")
for item in os.listdir(base + '/mashups')[:10]:
    print(item)

print("\n=== ESC-50-master ===")
for item in os.listdir(base + '/ESC-50-master')[:5]:
    print(item)

=== genres_stems ===
disco
metal
reggae
blues
rock
classical
jazz
hiphop
country
pop

=== mashups ===
song2501.wav
song0956.wav
song2874.wav
song1970.wav
song0718.wav
song1339.wav
song0060.wav
song1574.wav
song0602.wav
song2818.wav

=== ESC-50-master ===
meta
LICENSE
README.md
audio


In [5]:
os.system('git clone https://github.com/Photon-08/milestone-4-codebase.git /kaggle/working/milestone4')

for item in os.listdir('/kaggle/working/milestone4'):
    print(item)

Cloning into '/kaggle/working/milestone4'...


model_skeleton.py
README.md
.git
helper_functions.py


In [6]:
with open('/kaggle/working/milestone4/helper_functions.py', 'r') as f:
    print(f.read())

import torch
import numpy as np
import random
import torchaudio
import os
import glob
from pathlib import Path

# --- SET YOUR KAGGLE PATHS ---
INPUT_BASE = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'
WORKING_BASE = '/kaggle/working'

STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
OUTPUT_PATH = os.path.join(WORKING_BASE, 'synthetic_mashups/train')


def seed_everything(seed=42):
    """Locks all random seeds for absolute reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    # If using GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Forces deterministic algorithms
        torch.backends.cudnn.deterministic = True 
        torch.backends.cudnn.benchmark = False

# Execute immediately at the top of the script
seed_everything(42)







def generate_synthetic_dataset(stems

In [7]:
import torch
import numpy as np
import random
import torchaudio
import os
import glob
from pathlib import Path

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)
print("Seed set!")

Seed set!


In [8]:
INPUT_BASE = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
OUTPUT_PATH = '/kaggle/working/synthetic_mashups/train'

def generate_synthetic_dataset(stems_dir, noise_dir, output_dir, samples_per_genre=50, target_sr=22050, duration=30):
    genres = ["blues", "classical", "country", "disco", "hiphop",
              "jazz", "metal", "pop", "reggae", "rock"]
    target_length = target_sr * duration
    noise_files = glob.glob(os.path.join(noise_dir, '**', '*.wav'), recursive=True)
    
    for genre in genres:
        genre_out_dir = Path(output_dir) / genre
        genre_out_dir.mkdir(parents=True, exist_ok=True)
        song_folders = glob.glob(os.path.join(stems_dir, genre, '*'))
        
        if not song_folders:
            print(f"Warning: No songs found for genre {genre}")
            continue
        
        for i in range(samples_per_genre):
            chosen_songs = random.sample(song_folders, 4)
            stems = []
            stem_types = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
            
            for song, stem_type in zip(chosen_songs, stem_types):
                stem_path = os.path.join(song, stem_type)
                if os.path.exists(stem_path):
                    waveform, sr = torchaudio.load(stem_path)
                    if sr != target_sr:
                        resampler = torchaudio.transforms.Resample(sr, target_sr)
                        waveform = resampler(waveform)
                    if waveform.shape[1] > target_length:
                        waveform = waveform[:, :target_length]
                    elif waveform.shape[1] < target_length:
                        waveform = torch.nn.functional.pad(waveform, (0, target_length - waveform.shape[1]))
                    stems.append(waveform)
            
            if len(stems) == 4:
                mashup = torch.stack(stems).sum(dim=0)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                noise_file = random.choice(noise_files)
                noise, _ = torchaudio.load(noise_file)
                if noise.shape[1] > target_length:
                    noise = noise[:, :target_length]
                start_idx = random.randint(0, target_length - noise.shape[1])
                intensity = random.uniform(0.1, 0.4)
                mashup[:, start_idx:start_idx + noise.shape[1]] += (noise * intensity)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                out_path = genre_out_dir / f"mashup_{i:03d}.wav"
                torchaudio.save(str(out_path), mashup, target_sr)
        
        print(f"Done: {genre}")

generate_synthetic_dataset(STEMS_PATH, NOISE_PATH, OUTPUT_PATH, samples_per_genre=50)
print("Dataset generation complete!")

Done: blues
Done: classical
Done: country
Done: disco
Done: hiphop
Done: jazz
Done: metal
Done: pop
Done: reggae
Done: rock
Dataset generation complete!


In [14]:
wav_files = glob.glob('/kaggle/working/synthetic_mashups/train/**/*.wav', recursive=True)
print(f"Total .wav files: {len(wav_files)}")
# Answer = 10 genres × 50 samples = 500

Total .wav files: 500


In [15]:
sample_wav = wav_files[0]
waveform, sr = torchaudio.load(sample_wav)
print(f"Waveform shape: {tuple(waveform.shape)}")
# Expected: (1, 661500) = (1 channel, 22050 * 30)

Waveform shape: (2, 661500)


In [16]:
def extract_and_save_features(input_dir, output_dir, target_sr=22050):
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=target_sr, n_fft=2048, hop_length=512, n_mels=128
    )
    amplitude_to_db = torchaudio.transforms.AmplitudeToDB()
    wav_files = glob.glob(os.path.join(input_dir, '**', '*.wav'), recursive=True)
    
    if not wav_files:
        print(f"No .wav files found in {input_dir}")
        return
    
    for wav_path in wav_files:
        rel_path = os.path.relpath(wav_path, input_dir)
        out_path = Path(output_dir) / rel_path
        out_path = out_path.with_suffix('.pt')
        out_path.parent.mkdir(parents=True, exist_ok=True)
        waveform, sr = torchaudio.load(wav_path)
        mel_spec = mel_transform(waveform)
        mel_spec_db = amplitude_to_db(mel_spec)
        torch.save(mel_spec_db, out_path)
    
    print(f"Saved {len(wav_files)} feature files!")

extract_and_save_features(
    '/kaggle/working/synthetic_mashups/train',
    '/kaggle/working/features/train'
)

Saved 500 feature files!


In [17]:
pt_files = glob.glob('/kaggle/working/features/train/**/*.pt', recursive=True)
sample_tensor = torch.load(pt_files[0])
print(f"Tensor shape: {tuple(sample_tensor.shape)}")
# Expected: (1, 128, 1292)

Tensor shape: (2, 128, 1292)


In [18]:
import torch.nn as nn

class CRNN(nn.Module):
    def __init__(self, num_classes=10):
        super(CRNN, self).__init__()
        
        # CNN Block 1
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        # CNN Block 2
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        # Bidirectional LSTM
        # After 2 MaxPool2d(2): 128 -> 32 freq bins, 64 channels
        # input_size = 64 * 32 = 2048
        self.lstm = nn.LSTM(
            input_size=2048,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        
        # Final classifier
        # bidirectional = 64*2 = 128
        self.fc = nn.Linear(128, num_classes)
    
    def forward(self, x):
        # x: (batch, 1, 128, time)
        x = self.block1(x)   # (batch, 32, 64, time/2)
        x = self.block2(x)   # (batch, 64, 32, time/4)
        
        # Q4 answer - shape here before permute
        batch, channels, freq, time = x.shape
        
        # Reshape for LSTM: (batch, time, channels*freq)
        x = x.permute(0, 3, 1, 2)           # (batch, time, channels, freq)
        x = x.reshape(batch, time, -1)       # (batch, time, 2048)
        
        # LSTM
        x, _ = self.lstm(x)                  # (batch, time, 128)
        
        # Global Max Pooling
        x, _ = torch.max(x, dim=1)           # (batch, 128)
        
        # Final prediction
        x = self.fc(x)                       # (batch, 10)
        return x

model = CRNN(num_classes=10)
print(model)

CRNN(
  (block1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (lstm): LSTM(2048, 64, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=128, out_features=10, bias=True)
)


In [19]:
dummy_input = torch.randn(32, 1, 128, 1292)
x = model.block1(dummy_input)
x = model.block2(x)
print(f"Shape after 2nd MaxPool (before permute): {tuple(x.shape)}")
# Expected: (32, 64, 32, 323)

Shape after 2nd MaxPool (before permute): (32, 64, 32, 323)


In [20]:
lstm_params = sum(p.numel() for p in model.lstm.parameters() if p.requires_grad)
print(f"LSTM trainable parameters: {lstm_params}")

LSTM trainable parameters: 1082368


In [24]:
from torch.utils.data import Dataset, DataLoader

class MelDataset(Dataset):
    def __init__(self, features_dir):
        self.genres = ["blues", "classical", "country", "disco", "hiphop",
                       "jazz", "metal", "pop", "reggae", "rock"]
        self.label_map = {g: i for i, g in enumerate(self.genres)}
        self.files = []
        self.labels = []
        
        for genre in self.genres:
            pt_files = glob.glob(os.path.join(features_dir, genre, '*.pt'))
            for f in pt_files:
                self.files.append(f)
                self.labels.append(self.label_map[genre])
        
        print(f"Total samples: {len(self.files)}")
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        tensor = torch.load(self.files[idx])
        # Fix: Convert stereo to mono by averaging channels
        if tensor.shape[0] > 1:
            tensor = tensor.mean(dim=0, keepdim=True)
        return tensor, self.labels[idx]

full_dataset = MelDataset('/kaggle/working/features/train')

Total samples: 500


In [25]:
from torch.utils.data import random_split

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

Train: 400, Val: 100


In [26]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CRNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

num_epochs = 10

for epoch in range(num_epochs):
    # Training
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    
    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_correct += (predicted == labels).sum().item()
        train_total += labels.size(0)
    
    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    
    with torch.no_grad():
        for features, labels in val_loader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)
    
    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_loss/len(train_loader):.4f} "
          f"Train Acc: {100*train_correct/train_total:.2f}% "
          f"Val Loss: {val_loss/len(val_loader):.4f} "
          f"Val Acc: {100*val_correct/val_total:.2f}%")

Epoch [1/10] Train Loss: 2.2301 Train Acc: 22.00% Val Loss: 2.1758 Val Acc: 34.00%
Epoch [2/10] Train Loss: 1.9650 Train Acc: 39.25% Val Loss: 1.9755 Val Acc: 37.00%
Epoch [3/10] Train Loss: 1.7901 Train Acc: 50.00% Val Loss: 1.8590 Val Acc: 43.00%
Epoch [4/10] Train Loss: 1.6189 Train Acc: 59.75% Val Loss: 1.7233 Val Acc: 45.00%
Epoch [5/10] Train Loss: 1.4700 Train Acc: 63.00% Val Loss: 1.6149 Val Acc: 51.00%
Epoch [6/10] Train Loss: 1.3517 Train Acc: 71.25% Val Loss: 1.5566 Val Acc: 65.00%
Epoch [7/10] Train Loss: 1.2584 Train Acc: 75.25% Val Loss: 1.4664 Val Acc: 54.00%
Epoch [8/10] Train Loss: 1.1613 Train Acc: 80.25% Val Loss: 1.4552 Val Acc: 62.00%
Epoch [9/10] Train Loss: 1.0586 Train Acc: 80.00% Val Loss: 1.4368 Val Acc: 59.00%
Epoch [10/10] Train Loss: 0.9609 Train Acc: 87.25% Val Loss: 1.3313 Val Acc: 64.00%


In [29]:
dummy_input = torch.randn(32, 1, 128, 1292).to(device)
x = model.block1(dummy_input)
x = model.block2(x)
print(f"Q4 Answer - Shape after 2nd MaxPool: {tuple(x.shape)}")

Q4 Answer - Shape after 2nd MaxPool: (32, 64, 32, 323)


In [30]:
lstm_params = sum(p.numel() for p in model.lstm.parameters() if p.requires_grad)
print(f"Q5 Answer - LSTM trainable parameters: {lstm_params}")

Q5 Answer - LSTM trainable parameters: 1082368


In [31]:
import glob
import torch
import torchaudio

print("="*50)
print("MILESTONE 4 - ALL ANSWERS")
print("="*50)

# Q1
wav_files = glob.glob('/kaggle/working/synthetic_mashups/train/**/*.wav', recursive=True)
print(f"\nQ1 - Total .wav files: {len(wav_files)}")

# Q2
waveform, sr = torchaudio.load(wav_files[0])
print(f"Q2 - Waveform shape: {tuple(waveform.shape)}")

# Q3
pt_files = glob.glob('/kaggle/working/features/train/**/*.pt', recursive=True)
tensor = torch.load(pt_files[0])
print(f"Q3 - Tensor shape: {tuple(tensor.shape)}")

# Q4
dummy_input = torch.randn(32, 1, 128, 1292).to(device)
x = model.block1(dummy_input)
x = model.block2(x)
print(f"Q4 - Shape after 2nd MaxPool (before permute): {tuple(x.shape)}")

# Q5
lstm_params = sum(p.numel() for p in model.lstm.parameters() if p.requires_grad)
print(f"Q5 - LSTM trainable parameters: {lstm_params}")

print("\nQ6 - Moving a pattern up the Y-axis fundamentally changes")
print("     its pitch and instrument identity")
print("="*50)

MILESTONE 4 - ALL ANSWERS

Q1 - Total .wav files: 500
Q2 - Waveform shape: (2, 661500)
Q3 - Tensor shape: (2, 128, 1292)
Q4 - Shape after 2nd MaxPool (before permute): (32, 64, 32, 323)
Q5 - LSTM trainable parameters: 1082368

Q6 - Moving a pattern up the Y-axis fundamentally changes
     its pitch and instrument identity
